<a href="https://colab.research.google.com/github/AHichem/bridge-deterioration-nbi/blob/main/notebooks/02_sql_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
!pip uninstall -y -q ipython-sql
!pip install -q jupysql

from google.colab import drive
drive.mount('/content/drive')
DATA_DIR = '/content/drive/MyDrive/nbi'

import duckdb
con = duckdb.connect(f'{DATA_DIR}/nbi.duckdb')

%load_ext sql
%sql con
%config SqlMagic.displaylimit = 20

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
The sql extension is already loaded. To reload it, use:
  %reload_ext sql


In [6]:
%%sql
CREATE OR REPLACE TABLE inspections_raw AS
SELECT
    CAST(regexp_extract(filename, '(20\d{2})', 1) AS INTEGER) AS year,
    STATE_CODE_001, STRUCTURE_NUMBER_008, COUNTY_CODE_003, LAT_016, LONG_017,
    RECORD_TYPE_005A, OWNER_022, FUNCTIONAL_CLASS_026, YEAR_BUILT_027,
    YEAR_RECONSTRUCTED_106, SERVICE_ON_042A, STRUCTURE_KIND_043A, STRUCTURE_TYPE_043B,
    MAIN_UNIT_SPANS_045, STRUCTURE_LEN_MT_049, DECK_WIDTH_MT_052,
    ADT_029, PERCENT_ADT_TRUCK_109,
    DECK_COND_058, SUPERSTRUCTURE_COND_059, SUBSTRUCTURE_COND_060, CULVERT_COND_062,
    BRIDGE_CONDITION,
    DATE_OF_INSPECT_090, INSPECT_FREQ_MONTHS_091
FROM read_csv(
    [   -- 2016 à 2025 : nbi_2015.txt n'est pas un vrai fichier NBI (125 lignes, aucune colonne attendue)
        '/content/drive/MyDrive/nbi/raw/nbi_201[6-9].txt',
        '/content/drive/MyDrive/nbi/raw/nbi_202[0-9].txt'
    ],
    encoding = 'latin-1',    -- les fichiers NBI ne sont pas en UTF-8 (accents)
    delim = ',',
    quote = '''',            -- le qualificateur de texte est une apostrophe
    header = true,
    all_varchar = true,      -- tout en texte pour l'instant
    union_by_name = true,    -- aligne les colonnes par nom si l'ordre change selon l'année
    filename = true          -- ajoute le chemin du fichier, d'où on extrait l'année
);

Running query in 'DuckDBPyConnection'

InvalidInputException: Invalid Input Error: Error when sniffing file "".
It was not possible to automatically detect the CSV Parsing dialect/types
The search space used was:
Delimiter Candidates: ','
Quote/Escape Candidates: [''','(no escape)'],[''','''],[''','"'],[''','\']
Comment Candidates: '\0', '#'
Encoding: latin-1
Possible fixes:
* Disable the parser's strict mode (strict_mode=false) to allow reading rows that do not comply with the CSV standard.
* Make sure you are using the correct file encoding. If not, set it (e.g., encoding = 'utf-16').
* Delimiter is set to ','. Consider unsetting it.
* Quote is set to '''. Consider unsetting it.
* Set escape (e.g., escape='"')
* Set comment (e.g., comment='#')
* Set skip (skip=${n}) to skip ${n} lines at the top of the file
* Enable ignore errors (ignore_errors=true) to ignore potential errors
* Enable null padding (null_padding=true) to pad missing columns with NULL values
* Check you are using the correct file compression, otherwise set

In [ ]:
%%sql
SELECT year, COUNT(*) AS n_rows
FROM inspections_raw
GROUP BY year
ORDER BY year;